In [1]:
from __future__ import print_function

import math
import numpy as np
import numpy.linalg as nla
import pandas as pd
from pathlib import Path
import openpyxl
import re
import six
import duckdb
from os.path import join
import tensorflow as tf
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from keras_tuner import HyperParameters

In [2]:
def flatten_header_rows(file_path, header_rows):
	xl = pd.ExcelFile(file_path)
	xl_sheets = xl.sheet_names
	cleaned_dfs = []
	for sheet in xl_sheets:

		header_df = pd.read_excel(file_path, sheet_name=sheet, nrows=3, header=None)

		header_df = header_df.ffill(axis=1).fillna('')

		flat_columns = []
		for col in header_df.columns:
			flat_header = ""
			for level in range(header_rows):
				level_text = str(header_df.iloc[level, col]).strip()
				if level_text != '':
					flat_header += level_text + ' '
			
			flat_columns.append(flat_header.strip())

		final_unique_columns = []
		col_counts = {}
		for col in flat_columns:
			if col not in col_counts:
				col_counts[col] = 0
				final_unique_columns.append(col)
			else:
				col_counts[col] += 1
				final_unique_columns.append(f"{col}_{col_counts[col]}")

		data_df = pd.read_excel(file_path, sheet_name=sheet, skiprows=header_rows, header=None)

		data_df = data_df.iloc[:, :len(final_unique_columns)]
		data_df.columns = final_unique_columns
		data_df['Source_Sheet'] = sheet
		cleaned_dfs.append(data_df)

	final_cleaned_df = pd.concat(cleaned_dfs, ignore_index= True)
	return final_cleaned_df


def clean_column_headers(df):
	df.columns = df.columns.astype(str).str.replace(r'\s+',' ', regex = True)
	return df
	



def print_df_detail(dataframe, df_name = 'Dataframe', head_rows = 5):
	print(f"{df_name} shape: {dataframe.shape}")
	index_length = len(str(len(dataframe.columns)))
	values_length = len(str(len(dataframe)))
	columns_width = max(len(str(column)) for column in dataframe.columns)
	
	for index, column in enumerate(dataframe.columns):
		print(f"Column {index:0{index_length}d}: {column:>{columns_width}}  │  {len(dataframe[column].unique()):{values_length}d} unique values  │  dtype:{str(dataframe[column].dtype):>8}  │  nulls:{dataframe[column].isna().sum():{values_length}d}  │  zeroes:{(dataframe[column] == 0).sum():{values_length}d} | first 5 unique values: {dataframe[column].unique()[:5].tolist()}") 

	display(dataframe.head(head_rows))

In [3]:
current_dir = Path.cwd()
data_dir = current_dir.joinpath("data")
print(data_dir)

income_df_raw = pd.read_csv(data_dir.joinpath("2023/ACSST5Y2023.S1901-Data.csv"), sep=",", header=1, encoding='latin-1') #S1901 2023 Income in the Past 12 Months (ACS 5-Year Estimate - In 2023 Inflation-Adjusted Dollars)() - found here: https://data.census.gov/table/ACSST5Y2023.S1901?t=Income+and+Poverty&g=010XX00US$8600000&y=2023
race_df_raw = pd.read_csv(data_dir.joinpath("2023/ACSDT5Y2023.B02001-Data.csv"), sep=",", header=1, encoding='latin-1') #B02001 2023 Racial Populations in the Past 12 Months (ACS 5-Year Estimate)() - found here: https://data.census.gov/table/ACSDT5Y2023.B02001?t=Race+and+Ethnicity&g=010XX00US$8600000&y=2023
business_df_raw = pd.read_csv(data_dir.joinpath("2023/zbp23detail.zip"), sep=",", encoding='latin-1') #2023 Business Employment Data - found here: https://www2.census.gov/programs-surveys/cbp/datasets/2023/zbp23detail.zip
iou_df_raw = pd.read_csv(data_dir.joinpath("2023/iou_zipcodes_2023.csv"), sep=",", encoding='latin-1') #2023 Investor Owned Utilities - found here: https://data.openei.org/files/6225/iou_zipcodes_2023.csv
non_iou_df_raw = pd.read_csv(data_dir.joinpath("2023/non_iou_zipcodes_2023.csv"), sep=",", encoding='latin-1') #2023 Non Investor Owned Utilities - found here: https://data.openei.org/files/6225/non_iou_zipcodes_2023.csv


power_plant_generation_df_raw = clean_column_headers(pd.read_excel(data_dir.joinpath("2023/EIA923_Schedules_2_3_4_5_M_12_2023_Final_Revision.xlsx"), sheet_name='Page 1 Generation and Fuel Data', skiprows=5)) #Contains Power Plant generation for 2023 - EIA923_Schedules_2_3_4_5_M_12_2023_Final_Revision.xlsk file in 2023 ZIP found here: https://www.eia.gov/electricity/data/eia923/

utility_sales_df_raw = clean_column_headers(flatten_header_rows(file_path = data_dir.joinpath("2023/Sales_Ult_Cust_2023.xlsx"), header_rows = 3)) #Contains Utility Sales Data for 2023 - Sales_Ult_Cust_2023.xlsx file in 2023 zip found here: https://www.eia.gov/electricity/data/eia861/

region_source_mix_df_raw = clean_column_headers(pd.read_excel(data_dir.joinpath("2023/egrid2023_data_rev2.xlsx"), sheet_name='SRL23', skiprows=1)) #Regional Power Source Mix Data for 2023- found here: https://www.epa.gov/system/files/documents/2025-06/egrid2023_data_rev2.xlsx
region_zip_mapping_df_raw = clean_column_headers(pd.read_excel(data_dir.joinpath("2023/power_profiler_zipcode_tool_v14.3.xlsx"), sheet_name='Zip-subregion')) #2023 Racial Populations in the Past 12 Months (ACS 5-Year Estimate)() - found here: https://www.epa.gov/system/files/documents/2025-06/power_profiler_zipcode_tool_v14.3.xlsx




c:\Users\Roe 2019\Documents\GitHub\watts-the-cost-DATASCI-207\data


In [4]:
################
#Clean income_df
################
#Make copies to play with
income_df = income_df_raw.copy()

print_df_detail(income_df, "income_df")

income_df shape: (33772, 131)
Column 000:                                                                                           Geography  │  33772 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0 | first 5 unique values: ['860Z200US00601', '860Z200US00602', '860Z200US00603', '860Z200US00606', '860Z200US00610']
Column 001:                                                                                Geographic Area Name  │  33772 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0 | first 5 unique values: ['ZCTA5 00601', 'ZCTA5 00602', 'ZCTA5 00603', 'ZCTA5 00606', 'ZCTA5 00610']
Column 002:                                                                         Estimate!!Households!!Total  │  10967 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:  944 | first 5 unique values: [5611, 12546, 19537, 1871, 8838]
Column 003:                                                                  Margin of Error!!Households!!Total  │   1148 unique va

,Geography,Geographic Area Name,Estimate!!Households!!Total,Margin of Error!!Households!!Total,"Estimate!!Households!!Total!!Less than $10,000","Margin of Error!!Households!!Total!!Less than $10,000","Estimate!!Households!!Total!!$10,000 to $14,999","Margin of Error!!Households!!Total!!$10,000 to $14,999","Estimate!!Households!!Total!!$15,000 to $24,999","Margin of Error!!Households!!Total!!$15,000 to $24,999",...,Margin of Error!!Nonfamily households!!Median income (dollars),Estimate!!Nonfamily households!!Mean income (dollars),Margin of Error!!Nonfamily households!!Mean income (dollars),Estimate!!Nonfamily households!!PERCENT ALLOCATED!!Household income in the past 12 months,Margin of Error!!Nonfamily households!!PERCENT ALLOCATED!!Household income in the past 12 months,Estimate!!Nonfamily households!!PERCENT ALLOCATED!!Family income in the past 12 months,Margin of Error!!Nonfamily households!!PERCENT ALLOCATED!!Family income in the past 12 months,Estimate!!Nonfamily households!!PERCENT ALLOCATED!!Nonfamily income in the past 12 months,Margin of Error!!Nonfamily households!!PERCENT ALLOCATED!!Nonfamily income in the past 12 months,Unnamed: 130
0,860Z200US00601,ZCTA5 00601,5611,258,24.0,3.2,15.8,3.6,25.7,4.1,...,1623,17435,2864,(X),(X),(X),(X),14.1,(X),NaN
1,860Z200US00602,ZCTA5 00602,12546,510,22.6,2.9,11.8,2.0,20.7,2.5,...,1833,17806,2205,(X),(X),(X),(X),19.4,(X),NaN
2,860Z200US00603,ZCTA5 00603,19537,671,28.5,2.5,12.2,1.9,18.0,1.9,...,1199,18433,2062,(X),(X),(X),(X),35.3,(X),NaN
3,860Z200US00606,ZCTA5 00606,1871,192,23.6,5.4,16.6,6.1,21.2,5.8,...,4018,18057,3134,(X),(X),(X),(X),11.0,(X),NaN
4,860Z200US00610,ZCTA5 00610,8838,459,18.7,2.9,12.5,3.0,21.9,3.4,...,2299,19469,2506,(X),(X),(X),(X),11.5,(X),NaN


In [5]:
income_df_short = income_df.iloc[:,:29]

#Gather ZIP from Geo Name
income_df_short.insert(0,'zip',income_df_short['Geographic Area Name'].astype(str).str[-5:])

#Drop Geo and Geo Name
income_df_short.drop(columns=['Geography','Geographic Area Name'], axis=1, inplace=True)

rename_substring_map ={
    'Estimate':'',
	'Margin of Error':'moe',
	'Total':'',
    '!!':'_',
	'(dollars)':'',
    ' ':'_',
	',999':'k',
	',000':'k'
	
    
}
rename_dict = {}
for column in income_df_short.columns.copy():
	new_column_name = column
	for old_substr, new_substr in rename_substring_map.items():
		if old_substr in new_column_name:
			new_column_name = new_column_name.replace(old_substr, new_substr)
	while '__' in new_column_name:
		new_column_name = new_column_name.replace('__','_')		
	new_column_name = new_column_name.strip('_')
	rename_dict[column] = new_column_name.lower()


income_df_short.rename(columns=rename_dict,inplace=True)

income_df_short = income_df_short.apply(pd.to_numeric, errors='coerce')


income_df_final = income_df_short.dropna()

print_df_detail(income_df_final)




Dataframe shape: (30507, 28)
Column 00:                                                                 zip  │  30507 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0 | first 5 unique values: [601, 602, 603, 606, 610]
Column 01:                                                          households  │  10947 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0 | first 5 unique values: [5611, 12546, 19537, 1871, 8838]
Column 02:                                                      moe_households  │   1146 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0 | first 5 unique values: [258, 510, 671, 192, 459]
Column 03:                                           households_less_than_$10k  │    347 unique values  │  dtype: float64  │  nulls:    0  │  zeroes: 3177 | first 5 unique values: [24.0, 22.6, 28.5, 23.6, 18.7]
Column 04:                                       moe_households_less_than_$10k  │    610 unique values  │  dtype: float64  │  nulls:   

,zip,households,moe_households,households_less_than_$10k,moe_households_less_than_$10k,households_$10k_to_$14k,moe_households_$10k_to_$14k,households_$15k_to_$24k,moe_households_$15k_to_$24k,households_$25k_to_$34k,...,moe_households_$100k_to_$149k,households_$150k_to_$199k,moe_households_$150k_to_$199k,households_$200k_or_more,moe_households_$200k_or_more,households_median_income,moe_households_median_income,households_mean_income,moe_households_mean_income,households_percent_allocated_household_income_in_the_past_12_months
0,601,5611,258,24.0,3.2,15.8,3.6,25.7,4.1,12.7,...,1.1,0.0,0.9,0.3,0.5,18571.0,1365.0,24781.0,1987.0,15.4
1,602,12546,510,22.6,2.9,11.8,2.0,20.7,2.5,15.6,...,0.8,0.7,0.5,0.4,0.4,21702.0,2026.0,30378.0,2097.0,20.6
2,603,19537,671,28.5,2.5,12.2,1.9,18.0,1.9,10.4,...,1.0,1.6,0.7,0.8,0.4,19243.0,1444.0,32478.0,1777.0,44.3
3,606,1871,192,23.6,5.4,16.6,6.1,21.2,5.8,18.2,...,0.7,0.0,2.8,0.0,2.8,20226.0,3619.0,25125.0,3029.0,17.3
4,610,8838,459,18.7,2.9,12.5,3.0,21.9,3.4,15.1,...,0.9,0.5,0.5,1.9,1.5,23732.0,1614.0,34513.0,4140.0,18.6


In [6]:
################
#Clean race_df
################

#Make copies to play with
race_df = race_df_raw.copy()

race_df = race_df.iloc[:,:22]

#Gather ZIP from Geo Name
race_df.insert(0,'zip',race_df['Geographic Area Name'].astype(str).str[-5:])

#Drop Geo and Geo Name
race_df.drop(columns=['Geography','Geographic Area Name'], axis=1, inplace=True)

rename_substring_map ={
    'Estimate':'',
	'Margin of Error':'moe',
	'Total:':'',
    '!!':'_',
	'(dollars)':'',
    ' ':'_',
	',999':'k',
	',000':'k'
	
    
}
rename_dict = {}
for column in race_df.columns.copy():
	new_column_name = column
	for old_substr, new_substr in rename_substring_map.items():
		if old_substr in new_column_name:
			new_column_name = new_column_name.replace(old_substr, new_substr)
	while '__' in new_column_name:
		new_column_name = new_column_name.replace('__','_')		
	new_column_name = new_column_name.strip('_')
	rename_dict[column] = new_column_name.lower()


race_df.rename(columns=rename_dict,inplace=True)

race_df = race_df.apply(pd.to_numeric, errors='coerce')

race_df.rename(columns={'':'all','moe':'moe_all'}, inplace=True)


race_df_final = race_df.dropna()

print_df_detail(race_df_final, "race_df")

race_df shape: (33762, 21)
Column 00:                                                                                 zip  │  33762 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0 | first 5 unique values: [601, 602, 603, 606, 610]
Column 01:                                                                                 all  │  15601 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:  591 | first 5 unique values: [16721, 37510, 48317, 5435, 25413]
Column 02:                                                                             moe_all  │   2932 unique values  │  dtype: float64  │  nulls:    0  │  zeroes:    0 | first 5 unique values: [477.0, 263.0, 1021.0, 331.0, 368.0]
Column 03:                                                                         white_alone  │  13585 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:  740 | first 5 unique values: [13904, 13781, 35550, 3697, 6582]
Column 04:                                               

,zip,all,moe_all,white_alone,moe_white_alone,black_or_african_american_alone,moe_black_or_african_american_alone,american_indian_and_alaska_native_alone,moe_american_indian_and_alaska_native_alone,asian_alone,...,native_hawaiian_and_other_pacific_islander_alone,moe_native_hawaiian_and_other_pacific_islander_alone,some_other_race_alone,moe_some_other_race_alone,two_or_more_races:,moe_two_or_more_races:,two_or_more_races:_two_races_including_some_other_race,moe_two_or_more_races:_two_races_including_some_other_race,"two_or_more_races:_two_races_excluding_some_other_race,_and_three_or_more_races","moe_two_or_more_races:_two_races_excluding_some_other_race,_and_three_or_more_races"
0,601,16721,477.0,13904,666,314,182,7,13,19,...,0,24,1120,369,1357,453,1235,428,122,142
1,602,37510,263.0,13781,1403,520,366,73,104,44,...,0,32,1732,593,21360,1486,3688,900,17672,1354
2,603,48317,1021.0,35550,1510,1572,399,32,38,8,...,0,32,6231,759,4924,973,3983,893,941,337
3,606,5435,331.0,3697,440,12,16,0,21,15,...,0,21,1332,275,379,199,339,197,40,61
4,610,25413,368.0,6582,1030,525,445,1,4,0,...,0,28,2437,673,15868,1184,4387,846,11481,1416


In [7]:
################
#Clean business_df
################

#Make copies to play with
business_df = business_df_raw.copy()

#Gather Business Sector from NAICS
business_df.insert(1,'sector',business_df['naics'].astype(str).str[:2])

#Remove ZIP name, city, city_name, state abbrev, NAICS
business_df.drop(columns=['name','city','stabbr','cty_name','naics'], axis=1, inplace=True)



#replace missing nujmbers with 0
business_df = business_df.apply(pd.to_numeric, errors='coerce')
business_df = business_df.fillna(0)


print_df_detail(business_df, "business_df")


business_df shape: (2974116, 12)
Column 00:      zip  │    34954 unique values  │  dtype:   int64  │  nulls:      0  │  zeroes:      0 | first 5 unique values: [501, 1001, 1002, 1003, 1004]
Column 01:   sector  │       25 unique values  │  dtype: float64  │  nulls:      0  │  zeroes:  34954 | first 5 unique values: [0.0, 22.0, 23.0, 31.0, 32.0]
Column 02:      est  │     2048 unique values  │  dtype:   int64  │  nulls:      0  │  zeroes:      0 | first 5 unique values: [5, 460, 3, 49, 7]
Column 03:      n<5  │     1338 unique values  │  dtype: float64  │  nulls:      0  │  zeroes:1157095 | first 5 unique values: [5.0, 217.0, 0.0, 32.0, 6.0]
Column 04:     n5_9  │      498 unique values  │  dtype: float64  │  nulls:      0  │  zeroes:2299985 | first 5 unique values: [0.0, 79.0, 8.0, 6.0, 4.0]
Column 05:   n10_19  │      402 unique values  │  dtype: float64  │  nulls:      0  │  zeroes:2514368 | first 5 unique values: [0.0, 73.0, 3.0, 8.0, 7.0]
Column 06:   n20_49  │      340 unique valu

,zip,sector,est,n<5,n5_9,n10_19,n20_49,n50_99,n100_249,n250_499,n500_999,n1000
0,501,0.0,5,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1001,0.0,460,217.0,79.0,73.0,53.0,24.0,13.0,0.0,0.0,0.0
2,1001,22.0,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1001,22.0,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1001,23.0,49,32.0,8.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0


In [8]:
#####################################################
#USE THIS FOR ALL SECTORS SUMMED - Results in 11 columns
#####################################################
business_df_final = business_df.drop(columns=['sector'],axis=1).groupby(['zip'], as_index=False).sum()


print_df_detail(business_df_final, "business_df_final")

business_df_final shape: (34954, 11)
Column 00:      zip  │  34954 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0 | first 5 unique values: [501, 1001, 1002, 1003, 1004]
Column 01:      est  │   6305 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0 | first 5 unique values: [5, 2300, 2701, 62, 410]
Column 02:      n<5  │   4335 unique values  │  dtype: float64  │  nulls:    0  │  zeroes: 2389 | first 5 unique values: [5.0, 978.0, 1271.0, 20.0, 0.0]
Column 03:     n5_9  │   1755 unique values  │  dtype: float64  │  nulls:    0  │  zeroes:10229 | first 5 unique values: [0.0, 254.0, 438.0, 21.0, 25.0]
Column 04:   n10_19  │   1379 unique values  │  dtype: float64  │  nulls:    0  │  zeroes:13906 | first 5 unique values: [0.0, 226.0, 333.0, 8.0, 73.0]
Column 05:   n20_49  │   1116 unique values  │  dtype: float64  │  nulls:    0  │  zeroes:17113 | first 5 unique values: [0.0, 183.0, 145.0, 13.0, 36.0]
Column 06:   n50_99  │    476 unique values  │  dtype: 

,zip,est,n<5,n5_9,n10_19,n20_49,n50_99,n100_249,n250_499,n500_999,n1000
0,501,5,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1001,2300,978.0,254.0,226.0,183.0,50.0,17.0,0.0,0.0,0.0
2,1002,2701,1271.0,438.0,333.0,145.0,25.0,0.0,0.0,0.0,0.0
3,1003,62,20.0,21.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1004,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
#####################################################
#USE THIS FOR DATA BY SECTOR - Results in 75 columns
#####################################################


#business_df2 = business_df.groupby(['zip','sector'], as_index=False).sum()
#business_df2.insert(10,'n<500',business_df2.iloc[:,3:10].sum(axis=1))
#business_df2.drop(columns=business_df2.columns[3:10], inplace=True)


#business_df_melted = business_df2.melt(
#	id_vars =['zip','sector'],
#	value_vars = ['n<500','n500_999','n1000'],
#	var_name ='size_class',
#    value_name ='estabs'
#)

#business_df_melted['sector_size'] = business_df_melted['sector'].astype(int).astype(str) + ' ' + business_df_melted['size_class'].astype(str)

#business_df_final2 = business_df_melted.pivot_table(
#	index = 'zip',
#	columns = 'sector_size',
#	values ='estabs',
#	aggfunc = 'sum',
#	fill_value=0
#)

#business_df_final.reset_index()
#business_df_final.columns.name = None

#print_df_detail(business_df_final2, "business_df_final2")







In [10]:
################
#Clean iou_df
################

#Make copies to play with
iou_df = iou_df_raw.copy()

print_df_detail(iou_df, "iou_df")

iou_df shape: (52074, 9)
Column 0:          zip  │  31542 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0 | first 5 unique values: [85321, 36560, 36513, 36280, 35473]
Column 1:        eiaid  │    142 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0 | first 5 unique values: [176, 195, 213, 219, 392]
Column 2: utility_name  │    142 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0 | first 5 unique values: ['Ajo Improvement Co', 'Alabama Power Co', 'Alaska Electric Light & Power Co.', 'Alaska Power and Telephone Co', 'Alpena Power Co']
Column 3:        state  │     50 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0 | first 5 unique values: ['AZ', 'AL', 'AK', 'MI', 'IA']
Column 4: service_type  │      2 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0 | first 5 unique values: ['Bundled', 'Delivery']
Column 5:    ownership  │      1 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0 | first

,zip,eiaid,utility_name,state,service_type,ownership,comm_rate,ind_rate,res_rate
0,85321,176,Ajo Improvement Co,AZ,Bundled,Investor Owned,0.103993,0.000000,0.117085
1,36560,195,Alabama Power Co,AL,Bundled,Investor Owned,0.140915,0.076707,0.159023
2,36513,195,Alabama Power Co,AL,Bundled,Investor Owned,0.140915,0.076707,0.159023
3,36280,195,Alabama Power Co,AL,Bundled,Investor Owned,0.140915,0.076707,0.159023
4,35473,195,Alabama Power Co,AL,Bundled,Investor Owned,0.140915,0.076707,0.159023


In [11]:
################
#Clean non_iou_df
################

#Make copies to play with
non_iou_df = non_iou_df_raw.copy()

print_df_detail(non_iou_df, "non_iou_df")

non_iou_df shape: (28068, 9)
Column 0:          zip  │  19259 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0 | first 5 unique values: [39730, 38858, 21864, 21824, 21866]
Column 1:        eiaid  │   1060 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0 | first 5 unique values: [55, 84, 108, 123, 155]
Column 2: utility_name  │   1060 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0 | first 5 unique values: ['City of Aberdeen - (MS)', 'A & N Electric Coop', 'Adams-Columbia Electric Coop', 'City of Adel- (GA)', 'Agralite Electric Coop']
Column 3:        state  │     49 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0 | first 5 unique values: ['MS', 'MD', 'VA', 'WI', 'GA']
Column 4: service_type  │      3 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0 | first 5 unique values: ['Bundled', 'Energy', 'Delivery']
Column 5:    ownership  │      6 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:  

,zip,eiaid,utility_name,state,service_type,ownership,comm_rate,ind_rate,res_rate
0,39730,55,City of Aberdeen - (MS),MS,Bundled,Municipal,0.12279,0.056408,0.123420
1,38858,55,City of Aberdeen - (MS),MS,Bundled,Municipal,0.12279,0.056408,0.123420
2,21864,84,A & N Electric Coop,MD,Bundled,Cooperative,0.14049,0.000000,0.138776
3,21824,84,A & N Electric Coop,MD,Bundled,Cooperative,0.14049,0.000000,0.138776
4,21866,84,A & N Electric Coop,MD,Bundled,Cooperative,0.14049,0.000000,0.138776


In [21]:
################
#Clean power_plant_generation_df
################

#Make copies to play with

power_plant_generation_df = power_plant_generation_df_raw.copy()

#filter for columns we want
target_cols = ['Operator Id', 'MER Fuel Type Code', 'Net Generation (Megawatthours)']
gen_df_filtered = power_plant_generation_df[target_cols].copy()


eia_mer_decoder = {
'SUN': 'Solar PV and thermal',
'COL': 'Coal',
'DFO': 'Distillate Petroleum',
'GEO': 'Geothermal',
'HPS': 'Hydroelectric Pumped Storage',
'HYC': 'Hydroelectric Conventional',
'MLG': 'Biogenic Municipal Solid Waste and Landfill Gas',
'NG': 'Natural Gas',
'NUC': 'Nuclear',
'OOG': 'Other Gases',
'ORW': 'Other Renewables',
'OTH': 'Other (including nonbiogenic MSW)',
'PC': 'Petroleum Coke',
'RFO': 'Residual Petroleum',
'WND': 'Wind',
'WOC': 'Waste Coal',
'WOO': 'Waste Oil',
'WWW': 'Wood and Wood Waste',
}

gen_df_filtered['Fuel_Description'] = gen_df_filtered['MER Fuel Type Code'].map(eia_mer_decoder).fillna('Other Unknown')


#group by fuel type
grouped = gen_df_filtered.groupby(['Operator Id', 'Fuel_Description'])['Net Generation (Megawatthours)'].sum().reset_index()

gen_df_pivoted = grouped.pivot(index='Operator Id', columns = 'Fuel_Description', values = 'Net Generation (Megawatthours)').fillna(0)

gen_df_pivoted['Utl Total_Generation_MWh'] = gen_df_pivoted.sum(axis=1)

fuel_columns = [col for col in gen_df_pivoted.columns if col in eia_mer_decoder.values()]
for fuel in fuel_columns:
    gen_df_pivoted[fuel] = (gen_df_pivoted[fuel]/gen_df_pivoted['Utl Total_Generation_MWh']).fillna(0)
    gen_df_pivoted = gen_df_pivoted.rename(columns= {fuel: f"Utl {fuel}_Pct"})

gen_df_pivoted = gen_df_pivoted.reset_index()
gen_df_pivoted.columns.name = None


#Force Operator Id to Int
gen_df_pivoted['Operator Id'] = pd.to_numeric(gen_df_pivoted['Operator Id'], errors = 'coerce')
gen_df_pivoted = gen_df_pivoted.dropna()
gen_df_pivoted['Operator Id'] = gen_df_pivoted['Operator Id'].astype(int)


gen_df_final = gen_df_pivoted
print_df_detail(gen_df_final, "gen_df_final")

gen_df_final shape: (5176, 20)
Column 00:                                             Operator Id  │  5176 unique values  │  dtype:   int64  │  nulls:   0  │  zeroes:   0 | first 5 unique values: [7, 8, 25, 34, 35]
Column 01: Utl Biogenic Municipal Solid Waste and Landfill Gas_Pct  │    87 unique values  │  dtype: float64  │  nulls:   0  │  zeroes:4986 | first 5 unique values: [0.0, 0.42743972673520647, 0.4488673523158092, 0.4428241962167698, 0.44999899558514017]
Column 02:                                            Utl Coal_Pct  │   171 unique values  │  dtype: float64  │  nulls:   0  │  zeroes:5002 | first 5 unique values: [0.0, 0.9975554437781702, 0.341622744695377, 0.9868834678062339, 1.0]
Column 03:                            Utl Distillate Petroleum_Pct  │   569 unique values  │  dtype: float64  │  nulls:   0  │  zeroes:4346 | first 5 unique values: [0.0, 0.0024445562218298198, 1.0, 0.00011952152216909415, 0.00021252452665175136]
Column 04:                                      Ut

,Operator Id,Utl Biogenic Municipal Solid Waste and Landfill Gas_Pct,Utl Coal_Pct,Utl Distillate Petroleum_Pct,Utl Geothermal_Pct,Utl Hydroelectric Conventional_Pct,Utl Hydroelectric Pumped Storage_Pct,Utl Natural Gas_Pct,Utl Nuclear_Pct,Utl Other (including nonbiogenic MSW)_Pct,Utl Other Gases_Pct,Utl Other Renewables_Pct,Utl Petroleum Coke_Pct,Utl Residual Petroleum_Pct,Utl Solar PV and thermal_Pct,Utl Waste Coal_Pct,Utl Waste Oil_Pct,Utl Wind_Pct,Utl Wood and Wood Waste_Pct,Utl Total_Generation_MWh
0,7,0.0,0.000000,0.000000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,300668.00
1,8,0.0,0.000000,0.000000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,16833.38
2,25,0.0,0.000000,0.000000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,624116.00
3,34,0.0,0.000000,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5588.00
4,35,0.0,0.997555,0.002445,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,658054.00


In [13]:
################
#Clean utility_sales_df_raw
################

#Make copies to play with

utility_sales_df = utility_sales_df_raw.copy()

rename_substring_map ={
    'Utility Characteristics':'',
	'RESIDENTIAL':'Res',
	'COMMERCIAL':'Com',
    'INDUSTRIAL':'Ind',
	'Megawatthours':'MWh',
	'TOTAL': 'Total'
	
    
}
rename_dict = {}
for column in utility_sales_df.columns.copy():
	new_column_name = column
	for old_substr, new_substr in rename_substring_map.items():
		if old_substr in new_column_name:
			new_column_name = new_column_name.replace(old_substr, new_substr)
	while '__' in new_column_name:
		new_column_name = new_column_name.replace('__','_')		
	new_column_name = new_column_name.strip('_')
	rename_dict[column] = new_column_name.strip()

utility_sales_df.rename(columns=rename_dict,inplace=True)



#filter for columns we want
target_cols = ['Utility Number', 'Res Sales MWh', 'Com Sales MWh', 'Ind Sales MWh']

utility_sales_df = utility_sales_df[target_cols].copy()


#Force Utility Number to Int
utility_sales_df['Utility Number'] = pd.to_numeric(utility_sales_df['Utility Number'], errors = 'coerce')
utility_sales_df = utility_sales_df.dropna()
utility_sales_df['Utility Number'] = utility_sales_df['Utility Number'].astype(int)

sales_df_final = utility_sales_df


print_df_detail(sales_df_final)

Dataframe shape: (2832, 4)
Column 0: Utility Number  │  1538 unique values  │  dtype:   int64  │  nulls:   0  │  zeroes:   0 | first 5 unique values: [55, 84, 108, 113, 123]
Column 1:  Res Sales MWh  │  2334 unique values  │  dtype:  object  │  nulls:   0  │  zeroes: 395 | first 5 unique values: [32434, 2058, 355433, 275559, 83]
Column 2:  Com Sales MWh  │  2409 unique values  │  dtype:  object  │  nulls:   0  │  zeroes: 281 | first 5 unique values: [28398, 531, 173949, 89336, 0]
Column 3:  Ind Sales MWh  │  1625 unique values  │  dtype:  object  │  nulls:   0  │  zeroes: 992 | first 5 unique values: [121028, 0, 158475, 273957, 109555]


,Utility Number,Res Sales MWh,Com Sales MWh,Ind Sales MWh
0,55,32434,28398,121028
1,84,2058,531,0
2,84,355433,173949,158475
3,108,275559,89336,273957
4,113,83,0,0


In [22]:
################
#Clean region_source_mix
################

region_source_mix_df = region_source_mix_df_raw.copy()

target_col_indices = [1] + list(range(129, 140)) + list(range(158, 169))
region_mix_df = region_source_mix_df.iloc[:, target_col_indices]

rename_substring_map = {
    'PR':'Pct',
	'SUBRGN':'Sub-Region',
	'SR':'',
	'NB':'Non-Base ',	
	'CL': 'Regional Coal ',
	'OL': 'Regional Oil ',
	'GS': 'Regional Gas ',	
	'NC': 'Regional Nuclear ',
	'HY': 'Regional Hydroelectric ',
	'BM': 'Regional Biomass ',
	'GT': 'Regional Geothermal ',
	'WI': 'Regional Wind ',
	'SO': 'Regional Solar ',
	'OF': 'Regional Other Fossil ',
	'OP': 'Regional Other Unknown '
    
}
rename_dict = {}
for column in region_mix_df.columns.copy():
	new_column_name = column
	for old_substr, new_substr in rename_substring_map.items():
		if old_substr in new_column_name:
			new_column_name = new_column_name.replace(old_substr, new_substr)
	while '__' in new_column_name:
		new_column_name = new_column_name.replace('__','_')		
	new_column_name = new_column_name.strip('_')
	rename_dict[column] = new_column_name.strip()

region_mix_df.rename(columns=rename_dict,inplace=True)


region_mix_df_final = region_mix_df.dropna()
print_df_detail(region_mix_df_final)


Dataframe shape: (27, 23)
Column 00:                          Sub-Region  │  27 unique values  │  dtype:  object  │  nulls: 0  │  zeroes: 0 | first 5 unique values: ['AKGD', 'AKMS', 'AZNM', 'CAMX', 'ERCT']
Column 01:                   Regional Coal Pct  │  22 unique values  │  dtype: float64  │  nulls: 0  │  zeroes: 6 | first 5 unique values: [0.143, 0.0, 0.111, 0.021, 0.127]
Column 02:                    Regional Oil Pct  │  12 unique values  │  dtype: float64  │  nulls: 0  │  zeroes: 3 | first 5 unique values: [0.09, 0.246, 0.0, 0.005, 0.659]
Column 03:                    Regional Gas Pct  │  25 unique values  │  dtype: float64  │  nulls: 0  │  zeroes: 2 | first 5 unique values: [0.604, 0.065, 0.487, 0.429, 0.496]
Column 04:                Regional Nuclear Pct  │  18 unique values  │  dtype: float64  │  nulls: 0  │  zeroes:10 | first 5 unique values: [0.0, 0.18, 0.08, 0.085, 0.12]
Column 05:          Regional Hydroelectric Pct  │  22 unique values  │  dtype: float64  │  nulls: 0  │  

C:\Users\Roe 2019\AppData\Local\Temp\ipykernel_13408\3419845742.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  region_mix_df.rename(columns=rename_dict,inplace=True)


,Sub-Region,Regional Coal Pct,Regional Oil Pct,Regional Gas Pct,Regional Nuclear Pct,Regional Hydroelectric Pct,Regional Biomass Pct,Regional Wind Pct,Regional Solar Pct,Regional Geothermal Pct,...,Non-Base Regional Oil Pct,Non-Base Regional Gas Pct,Non-Base Regional Nuclear Pct,Non-Base Regional Hydroelectric Pct,Non-Base Regional Biomass Pct,Non-Base Regional Wind Pct,Non-Base Regional Solar Pct,Non-Base Regional Geothermal Pct,Non-Base Regional Other Fossil Pct,Non-Base Regional Other Unknown Pct
0,AKGD,0.143,0.090,0.604,0.000,0.139,0.008,0.016,0.000,0.000,...,0.148,0.677,0,0,0.010,0,0,0,0.000,0.000
1,AKMS,0.000,0.246,0.065,0.000,0.672,0.000,0.018,0.000,0.000,...,0.792,0.208,0,0,0.000,0,0,0,0.000,0.000
2,AZNM,0.111,0.000,0.487,0.180,0.026,0.003,0.077,0.082,0.036,...,0.000,0.799,0,0,0.003,0,0,0,0.000,0.000
3,CAMX,0.021,0.000,0.429,0.080,0.141,0.022,0.066,0.201,0.036,...,0.001,0.890,0,0,0.027,0,0,0,0.019,0.003
4,ERCT,0.127,0.000,0.496,0.085,0.001,0.002,0.226,0.059,0.000,...,0.001,0.755,0,0,0.002,0,0,0,0.015,0.000


In [15]:
################
#Clean region_zip_mapping
################

region_zip_mapping_df = region_zip_mapping_df_raw.copy()

target_col_indices = [0,1]
region_zip_mapping_df_final = region_zip_mapping_df.iloc[:, target_col_indices].dropna()

print_df_detail(region_zip_mapping_df_final)

Dataframe shape: (41588, 2)
Column 0:         zip  │  41588 unique values  │  dtype:   int64  │  nulls:    0  │  zeroes:    0 | first 5 unique values: [1, 2, 3, 4, 5]
Column 1: Subregion 1  │     27 unique values  │  dtype:  object  │  nulls:    0  │  zeroes:    0 | first 5 unique values: ['AKMS', 'AKGD', 'CAMX', 'AZNM', 'RMPA']


,zip,Subregion 1
0,1,AKMS
1,2,AKMS
2,3,AKMS
3,4,AKMS
4,5,AKMS


In [16]:
inc_cols = 'inc."' +'",\n		inc."'.join(income_df_final.columns) + '"'
bs_cols = 'bs."' +'",\n		bs."'.join(business_df_final.drop(columns='zip').columns) + '"'
race_cols = 'race."' +'",\n		race."'.join(race_df_final.drop(columns='zip').columns) + '"'
gen_cols = 'gen."' +'",\n		gen."'.join(gen_df_final.drop(columns='Operator Id').columns) + '"'
sales_cols = 'sales."' +'",\n		sales."'.join(sales_df_final.drop(columns='Utility Number').columns) + '"'
rmix_cols = 'rmix."' +'",\n		rmix."'.join(region_mix_df_final.drop(columns='Sub-Region').columns) + '"'
zmix_cols = 'zmix."' +'",\n		zmix."'.join(region_mix_df_final.drop(columns='Sub-Region').columns) + '"'


SQL = f"""
	WITH
    rate_type_key (rate_type, rate_type_index) as (
		values
			('commercial',0),
            ('industrial',1),
            ('residential',2),
    
    
    
    ),
    Utility_Data AS (
		SELECT
        iou.zip,
		iou.service_type,
        iou.ownership,
        iou.eiaid,
        rtk.rate_type_index,
        case
        	when rtk.rate_type = 'commercial' then iou.comm_rate
        	when rtk.rate_type = 'industrial' then iou.ind_rate
            when rtk.rate_type = 'residential' then iou.res_rate
            end AS rate

        FROM iou_df iou, rate_type_key rtk
        
        
        
        UNION ALL
	
		SELECT
        non_iou.zip,
		non_iou.service_type,
        non_iou.ownership,
        non_iou.eiaid,
		rtk.rate_type_index,
        case
        	when rtk.rate_type = 'commercial' then non_iou.comm_rate
        	when rtk.rate_type = 'industrial' then non_iou.ind_rate
            when rtk.rate_type = 'residential' then non_iou.res_rate
            end AS rate

        FROM non_iou_df non_iou, rate_type_key rtk
    ),
    
    zip_mix AS (
		SELECT
		rmap.zip,
		{rmix_cols}
		
		FROM region_mix_df_final rmix
		
		INNER JOIN region_zip_mapping_df_final rmap
		ON rmix."Sub-Region" = rmap."Subregion 1"
    ),
    
    
    COMBINED_DATA AS (
		SELECT
		{inc_cols},
        {race_cols},
		{bs_cols},
        {zmix_cols},
        {gen_cols},
        {sales_cols},
		ud.service_type,
		ud.ownership,
        ud.rate_type_index,
        ud.rate
		
    
    
    
    	FROM income_df_final inc

		INNER JOIN business_df_final bs
		ON bs.zip = inc.zip
        
        INNER JOIN race_df_final race
		ON race.zip = inc.zip
       
	   	INNER JOIN Utility_Data ud
	   	ON ud.zip = inc.zip
           
        INNER JOIN zip_mix zmix
        ON zmix.zip = inc.zip
        
        INNER JOIN gen_df_final gen
        ON gen."Operator Id" = ud.eiaid
        
        INNER JOIN sales_df_final sales
        ON sales."Utility Number" = ud.eiaid
           
           
        WHERE ud.rate is not null and ud.rate > 0
    )

    SELECT *
    
    
    FROM combined_data CD
    
    
"""

print(SQL)


df_clean = duckdb.sql(SQL).df()

print_df_detail(df_clean)


	WITH
    rate_type_key (rate_type, rate_type_index) as (
		values
			('commercial',0),
            ('industrial',1),
            ('residential',2),



    ),
    Utility_Data AS (
		SELECT
        iou.zip,
		iou.service_type,
        iou.ownership,
        iou.eiaid,
        rtk.rate_type_index,
        case
        	when rtk.rate_type = 'commercial' then iou.comm_rate
        	when rtk.rate_type = 'industrial' then iou.ind_rate
            when rtk.rate_type = 'residential' then iou.res_rate
            end AS rate

        FROM iou_df iou, rate_type_key rtk



        UNION ALL

		SELECT
        non_iou.zip,
		non_iou.service_type,
        non_iou.ownership,
        non_iou.eiaid,
		rtk.rate_type_index,
        case
        	when rtk.rate_type = 'commercial' then non_iou.comm_rate
        	when rtk.rate_type = 'industrial' then non_iou.ind_rate
            when rtk.rate_type = 'residential' then non_iou.res_rate
            end AS rate

        FROM non_iou_df non_iou, rate_type_ke

,zip,households,moe_households,households_less_than_$10k,moe_households_less_than_$10k,households_$10k_to_$14k,moe_households_$10k_to_$14k,households_$15k_to_$24k,moe_households_$15k_to_$24k,households_$25k_to_$34k,...,Wind_Pct,Wood and Wood Waste_Pct,Total_Generation_MWh,Res Sales MWh,Com Sales MWh,Ind Sales MWh,service_type,ownership,rate_type_index,rate
0,1002,9715,427,9.7,3.2,5.5,2.0,9.2,3.6,5.9,...,0.0,0.0,2062.0,4335004,7312220,2274974,Delivery,Investor Owned,2,0.142767
1,1005,1688,185,1.0,1.6,4.4,5.6,4.1,5.3,4.0,...,0.0,0.0,2062.0,4335004,7312220,2274974,Delivery,Investor Owned,2,0.142767
2,1007,6429,359,2.7,1.5,2.9,1.6,6.2,3.1,4.6,...,0.0,0.0,2062.0,4335004,7312220,2274974,Delivery,Investor Owned,2,0.142767
3,1009,367,151,10.6,12.7,4.6,7.8,14.2,21.2,9.5,...,0.0,0.0,2062.0,4335004,7312220,2274974,Delivery,Investor Owned,2,0.142767
4,1010,1479,110,2.6,2.2,3.9,4.4,5.8,3.4,5.3,...,0.0,0.0,2062.0,4335004,7312220,2274974,Delivery,Investor Owned,2,0.142767


In [17]:
import pyarrow
df_clean.to_parquet('data/cleaned_data.parquet', index=False, engine='pyarrow')

In [18]:
for column in df_clean.columns:
    print(column)

zip
households
moe_households
households_less_than_$10k
moe_households_less_than_$10k
households_$10k_to_$14k
moe_households_$10k_to_$14k
households_$15k_to_$24k
moe_households_$15k_to_$24k
households_$25k_to_$34k
moe_households_$25k_to_$34k
households_$35k_to_$49k
moe_households_$35k_to_$49k
households_$50k_to_$74k
moe_households_$50k_to_$74k
households_$75k_to_$99k
moe_households_$75k_to_$99k
households_$100k_to_$149k
moe_households_$100k_to_$149k
households_$150k_to_$199k
moe_households_$150k_to_$199k
households_$200k_or_more
moe_households_$200k_or_more
households_median_income
moe_households_median_income
households_mean_income
moe_households_mean_income
households_percent_allocated_household_income_in_the_past_12_months
all
moe_all
white_alone
moe_white_alone
black_or_african_american_alone
moe_black_or_african_american_alone
american_indian_and_alaska_native_alone
moe_american_indian_and_alaska_native_alone
asian_alone
moe_asian_alone
native_hawaiian_and_other_pacific_islander_a